# 02 · Quando i dati crescono: pandas, Polars e DuckDB a confronto

**Demo «Python e VS Code nei flussi ETL»**

Il flusso della demo elabora ventimila righe in meno di un secondo. Ma che cosa succede quando le righe diventano milioni o centinaia di milioni? In questo notebook poniamo la stessa domanda a tre strumenti diversi, su un file di alcuni milioni di righe, e misuriamo quanto tempo impiega ciascuno.

**Che cosa faremo, passo per passo**

1. Prepariamo l'ambiente e il file di prova.
2. Formuliamo la domanda a cui i tre strumenti dovranno rispondere.
3. Rispondiamo con **pandas**, lo strumento usato nel flusso della demo.
4. Rispondiamo con **Polars**.
5. Rispondiamo con **DuckDB**.
6. Verifichiamo che le tre risposte coincidano.
7. Cambiamo il formato del file, da CSV a **Parquet**, e ripetiamo le misure.
8. Leggiamo i risultati e ne ricaviamo delle indicazioni pratiche.

## Assunzioni

Elenchiamo ciò che diamo per vero in questo notebook.

1. **Ambiente.** Il notebook viene eseguito in VS Code con l'ambiente conda `etl_demo`, che contiene pandas, Polars, DuckDB e PyArrow. Per verificarlo basta lanciare dal terminale `python strumenti/verifica_ambiente.py`.
2. **File di prova.** Il file `dati/grandi_volumi/movimenti_grande.csv` è già stato generato con il comando `python strumenti/genera_file_grande.py`. Con le impostazioni di base contiene 5 milioni di righe e occupa circa 210 MB.
3. **Dati puliti.** A differenza dei file giornalieri della demo, il file grande non contiene errori voluti: qui misuriamo la velocità, non la qualità dei dati.
4. **Una sola macchina.** Confrontiamo strumenti che lavorano su un solo computer. Non trattiamo i sistemi distribuiti, cioè quelli che ripartiscono il lavoro su molti computer.
5. **Valore delle misure.** I tempi dipendono dalla macchina, dal file e dalla domanda posta. I numeri che otterremo valgono per questa prova: indicano un ordine di grandezza, non una classifica assoluta.
6. **Versioni.** Il notebook è stato scritto per pandas 3.0, Polars 2.0 e DuckDB 2.0. Usa soltanto istruzioni presenti anche nelle versioni 1.x di Polars e di DuckDB.

## Le parole che useremo

| Termine | Significato |
|---|---|
| **Motore di calcolo** | Il programma che esegue materialmente le operazioni sui dati: leggere, filtrare, unire, aggregare. pandas, Polars e DuckDB sono tre motori diversi. |
| **Core** | Un processore moderno contiene più unità di calcolo indipendenti, dette *core*. Un programma che ne usa uno solo lascia inutilizzati tutti gli altri. |
| **In memoria** | Un motore lavora «in memoria» quando, prima di elaborare i dati, li carica tutti nella memoria RAM del computer. Se i dati sono più grandi della memoria disponibile, il programma si ferma con un errore. |
| **Valutazione pigra** | Modo di lavorare in cui prima si *descrive* l'intera elaborazione e solo alla fine la si *esegue*. Conoscendo tutta la richiesta in anticipo, il motore può organizzarsi: ad esempio leggere dal file solo le colonne che servono davvero. In inglese si dice *lazy evaluation*. |
| **SQL** | Il linguaggio standard per interrogare i database. |
| **CSV** | Formato di testo «a righe»: ogni riga del file contiene tutti i campi di un record. Per leggere una sola colonna bisogna comunque scorrere l'intero file. |
| **Parquet** | Formato binario «a colonne»: i valori di ogni colonna sono memorizzati insieme, compressi e con il loro tipo. Per leggere una sola colonna basta leggere quella parte del file. |

## 1 · Prepariamo l'ambiente e il file di prova

In [ ]:
import sys
import time                 # per misurare i tempi
from pathlib import Path    # per indicare cartelle e file

import duckdb               # motore SQL che lavora direttamente sui file
import pandas as pd         # la libreria usata nel flusso della demo
import polars as pl         # libreria per tabelle di dati che usa tutti i core del processore

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Polars:", pl.__version__)
print("DuckDB:", duckdb.__version__)

In [ ]:
# Individuiamo la cartella principale del progetto (quella che contiene config.toml).
# Un notebook aperto in VS Code lavora di norma nella cartella in cui si trova: in quel
# caso la cartella del progetto è quella immediatamente superiore.
RADICE_PROGETTO = Path.cwd()
if not (RADICE_PROGETTO / "config.toml").exists():
    RADICE_PROGETTO = RADICE_PROGETTO.parent

FILE_GRANDE = RADICE_PROGETTO / "dati" / "grandi_volumi" / "movimenti_grande.csv"
FILE_PARQUET = RADICE_PROGETTO / "dati" / "grandi_volumi" / "movimenti_grande.parquet"
FILE_CONTI = RADICE_PROGETTO / "dati" / "input" / "conti.csv"

# Se il file grande non esiste ancora ci fermiamo con un messaggio che spiega che cosa fare
if not FILE_GRANDE.exists():
    raise FileNotFoundError(
        "Il file grande non esiste ancora. Generarlo dal terminale di VS Code con:\n"
        "    python strumenti/genera_file_grande.py"
    )

megabyte_csv = FILE_GRANDE.stat().st_size / 1024**2     # dimensione in megabyte
print(f"File: {FILE_GRANDE.name} - Dimensione: {megabyte_csv:.0f} MB")

Guardiamo le prime righe del file, per ricordarci come è fatto. Con `nrows=5` chiediamo a pandas di leggere solo cinque righe: l'operazione è immediata anche su un file enorme.

In [ ]:
pd.read_csv(FILE_GRANDE, sep=";", nrows=5)

## 2 · La domanda

Porremo a tutti e tre gli strumenti la stessa domanda, tipica di un flusso ETL:

> **Per ogni filiale e per ogni causale, quanti movimenti in euro ci sono stati e per quale importo totale, considerando soltanto i conti attivi?**

Per rispondere servono quattro operazioni, le stesse che in uno strumento grafico corrisponderebbero a quattro blocchi collegati fra loro:

1. **filtrare** i movimenti, tenendo quelli in euro;
2. **unire** i movimenti all'anagrafica dei conti, per conoscere la filiale e lo stato di ogni conto, tenendo i soli conti attivi;
3. **raggruppare** per filiale e causale;
4. **aggregare**, cioè contare i movimenti e sommare gli importi di ogni gruppo.

### Due accorgimenti per un confronto equo

**Primo accorgimento.** Il sistema operativo conserva in memoria i file letti di recente. Senza precauzioni, lo strumento provato per primo pagherebbe da solo il tempo di lettura dal disco e gli altri troverebbero il file già pronto. Per questo leggiamo il file una volta «a vuoto» prima di cominciare.

In [ ]:
# Leggiamo il file a blocchi di 64 MB, senza farne nulla, finché non è finito.
# "rb" significa lettura binaria: i byte vengono letti così come sono, senza interpretarli.
with open(FILE_GRANDE, "rb") as file_aperto:
    while file_aperto.read(64 * 1024 * 1024):
        pass

print("File letto una volta: ora i tre strumenti partono alla pari.")

**Secondo accorgimento.** Misuriamo tutte le prove nello stesso modo, con una piccola funzione che fa da cronometro: fa partire il tempo, esegue la prova, ferma il tempo e annota il risultato.

In [ ]:
tempi = {}   # dizionario in cui annotiamo i secondi impiegati da ogni prova


def cronometra(nome_prova, funzione, formato):
    '''
    Esegue una prova misurando il tempo che impiega.

    nome_prova   l'etichetta con cui annotiamo il tempo
    funzione     la funzione da eseguire; la passiamo SENZA parentesi, perché non
                 vogliamo eseguirla subito: sarà cronometra a chiamarla
    formato      "csv" oppure "parquet": il tipo di file da cui leggere i movimenti
    '''
    inizio = time.perf_counter()
    risultato = funzione(formato)
    tempi[nome_prova] = time.perf_counter() - inizio
    print(f"{nome_prova:<20} {tempi[nome_prova]:7.2f} secondi")
    return risultato

## 3 · La risposta con pandas

pandas è un motore **in memoria** che usa **un solo core**. Per rispondere deve prima caricare il file nella memoria RAM, poi eseguire le operazioni una dopo l'altra.

Gli diamo comunque un aiuto, come farebbe chi lo conosce bene: con `usecols` gli chiediamo di caricare soltanto le quattro colonne che servono alla domanda, invece di tutte e sei.

In [ ]:
COLONNE_UTILI = ["id_conto", "importo", "divisa", "codice_causale"]


def con_pandas(formato):
    # 1) Carichiamo in memoria le colonne che servono
    if formato == "csv":
        movimenti = pd.read_csv(FILE_GRANDE, sep=";", usecols=COLONNE_UTILI)
    else:
        movimenti = pd.read_parquet(FILE_PARQUET, columns=COLONNE_UTILI)
    conti = pd.read_csv(FILE_CONTI, sep=";", usecols=["id_conto", "codice_filiale", "stato"])

    # 2) Filtriamo: movimenti in euro, conti attivi
    in_euro = movimenti[movimenti["divisa"] == "EUR"]
    conti_attivi = conti[conti["stato"] == "ATTIVO"]

    # 3) Uniamo le due tabelle usando la colonna in comune, id_conto
    uniti = in_euro.merge(conti_attivi, on="id_conto", how="inner")

    # 4) Raggruppiamo e aggreghiamo, poi ordiniamo il risultato
    return (
        uniti.groupby(["codice_filiale", "codice_causale"], as_index=False)
        .agg(numero_movimenti=("importo", "size"), totale=("importo", "sum"))
        .sort_values(["codice_filiale", "codice_causale"])
        .reset_index(drop=True)
    )


risultato_pandas = cronometra("pandas su CSV", con_pandas, "csv")
risultato_pandas.head()

## 4 · La risposta con Polars

Polars è una libreria per tabelle di dati, come pandas, ma costruita con due idee diverse:

- usa **tutti i core** del processore;
- adotta la **valutazione pigra**: con `scan_csv` non leggiamo il file, ci limitiamo a dichiarare da dove arrivano i dati. Poi descriviamo le operazioni. Solo l'istruzione finale, `collect`, mette in moto il lavoro. Conoscendo tutta la richiesta in anticipo, Polars legge dal file soltanto le colonne e le righe che servono.

Dalla versione 2.0, per impostazione predefinita, Polars elabora inoltre i dati un blocco alla volta, invece di caricare l'intero file in memoria. Lavorando a blocchi e su più core, l'ordine delle righe del risultato non è garantito: per questo chiudiamo la richiesta con un ordinamento esplicito (`sort`).

In [ ]:
def con_polars(formato):
    # 1) Dichiariamo le sorgenti: le istruzioni scan_* NON leggono ancora nulla
    if formato == "csv":
        movimenti = pl.scan_csv(FILE_GRANDE, separator=";")
    else:
        movimenti = pl.scan_parquet(FILE_PARQUET)
    conti = pl.scan_csv(FILE_CONTI, separator=";")

    # 2) Descriviamo la richiesta, un passo dopo l'altro.
    #    pl.col("divisa") significa "la colonna divisa".
    domanda = (
        movimenti.filter(pl.col("divisa") == "EUR")                                    # solo i movimenti in euro
        .join(conti.filter(pl.col("stato") == "ATTIVO"), on="id_conto", how="inner")   # unione con i conti attivi
        .group_by("codice_filiale", "codice_causale")                                  # un gruppo per filiale e causale
        .agg(
            pl.len().alias("numero_movimenti"),                                        # quante righe nel gruppo
            pl.col("importo").sum().alias("totale"),                                   # somma degli importi del gruppo
        )
        .sort("codice_filiale", "codice_causale")                                      # ordinamento esplicito
    )

    # 3) Solo adesso, con collect, Polars esegue davvero il lavoro
    return domanda.collect()


risultato_polars = cronometra("Polars su CSV", con_polars, "csv")
risultato_polars.head()

## 5 · La risposta con DuckDB

DuckDB è un motore di database che non richiede alcun server: vive dentro il nostro programma Python, come SQLite. La sua particolarità è che sa interrogare **direttamente i file**, come se fossero tabelle, usando il linguaggio **SQL**. Anche DuckDB usa tutti i core e legge soltanto ciò che serve.

Per chi conosce già l'SQL è la strada più breve: la domanda si scrive così come la scriveremmo per un database, mettendo il nome del file al posto del nome della tabella.

In [ ]:
def percorso_per_sql(percorso):
    '''
    Prepara un percorso per essere scritto dentro un comando SQL:
    - usa le barre in avanti, che funzionano anche su Windows;
    - raddoppia gli eventuali apostrofi, come richiede il linguaggio SQL
      (capita se una cartella si chiama, ad esempio, D'Angelo).
    '''
    return percorso.as_posix().replace("'", "''")


def con_duckdb(formato):
    # 1) Indichiamo la sorgente dei movimenti: una funzione che legge il file CSV oppure il file Parquet.
    #    Per il CSV dichiariamo il separatore di colonna (delim) e il fatto che la prima riga contiene
    #    i nomi delle colonne (header); i tipi delle colonne li riconosce DuckDB da solo.
    if formato == "csv":
        sorgente = f"read_csv('{percorso_per_sql(FILE_GRANDE)}', delim = ';', header = true)"
    else:
        sorgente = f"read_parquet('{percorso_per_sql(FILE_PARQUET)}')"

    # 2) Scriviamo la domanda in SQL. Le lettere "m" e "c" sono i nomi brevi (alias)
    #    che diamo alle due sorgenti: movimenti e conti.
    comando = f'''
        SELECT c.codice_filiale,
               m.codice_causale,
               COUNT(*)       AS numero_movimenti,
               SUM(m.importo) AS totale
        FROM {sorgente} AS m
        JOIN read_csv('{percorso_per_sql(FILE_CONTI)}', delim = ';', header = true) AS c
          ON m.id_conto = c.id_conto
        WHERE m.divisa = 'EUR'
          AND c.stato = 'ATTIVO'
        GROUP BY c.codice_filiale, m.codice_causale
        ORDER BY c.codice_filiale, m.codice_causale
    '''

    # 3) Eseguiamo il comando; .df() restituisce il risultato come DataFrame di pandas
    return duckdb.sql(comando).df()


risultato_duckdb = cronometra("DuckDB su CSV", con_duckdb, "csv")
risultato_duckdb.head()

## 6 · Le tre risposte coincidono?

Un confronto di velocità ha senso solo se i tre strumenti danno la **stessa risposta**. Lo verifichiamo confrontando i conteggi, che devono essere identici, e i totali, che devono coincidere a meno del centesimo. Una piccolissima differenza sulle ultime cifre decimali è normale: dipende dall'ordine in cui ciascun motore somma i numeri.

In [ ]:
# Portiamo il risultato di Polars in pandas, così possiamo confrontare i tre risultati nello stesso modo.
# to_dict(as_series=False) restituisce un dizionario "nome colonna -> elenco dei valori".
polars_in_pandas = pd.DataFrame(risultato_polars.to_dict(as_series=False))


def stessi_numeri(primo, secondo):
    '''Vero se le due tabelle hanno gli stessi conteggi e gli stessi totali, a meno del centesimo.'''
    stessi_conteggi = primo["numero_movimenti"].tolist() == secondo["numero_movimenti"].tolist()
    stessi_totali = (primo["totale"] - secondo["totale"]).abs().max() < 0.01
    return bool(stessi_conteggi and stessi_totali)


print("Righe del risultato:", len(risultato_pandas))
print("pandas e Polars danno la stessa risposta:", stessi_numeri(risultato_pandas, polars_in_pandas))
print("pandas e DuckDB danno la stessa risposta:", stessi_numeri(risultato_pandas, risultato_duckdb))

## 7 · Cambiamo formato: da CSV a Parquet

Finora abbiamo letto un file CSV, cioè un file di testo. Leggerlo è faticoso per qualunque motore: bisogna scorrerlo tutto, individuare i separatori e trasformare ogni pezzo di testo in un numero o in una data.

Il formato **Parquet** nasce per l'analisi dei dati e rimuove proprio queste fatiche: i dati sono memorizzati **per colonne**, già nel loro tipo e compressi. Chi ha bisogno di quattro colonne su sei legge solo quelle.

Convertiamo il file una volta sola, con DuckDB, e confrontiamo le dimensioni.

In [ ]:
inizio = time.perf_counter()

# Leggiamo tutto il CSV e lo riscriviamo in formato Parquet
sorgente_csv = f"read_csv('{percorso_per_sql(FILE_GRANDE)}', delim = ';', header = true)"
duckdb.sql(f"SELECT * FROM {sorgente_csv}").write_parquet(str(FILE_PARQUET))

print(f"Conversione completata in {time.perf_counter() - inizio:.1f} secondi")

megabyte_parquet = FILE_PARQUET.stat().st_size / 1024**2
print(f"File CSV:     {megabyte_csv:6.0f} MB")
print(f"File Parquet: {megabyte_parquet:6.0f} MB  (pari al {megabyte_parquet / megabyte_csv:.0%} del CSV)")

Ora poniamo di nuovo la stessa domanda ai tre strumenti, leggendo questa volta il file Parquet. Le funzioni sono quelle di prima: cambia soltanto il valore `"parquet"` che passiamo come formato.

In [ ]:
risultato = cronometra("pandas su Parquet", con_pandas, "parquet")
risultato = cronometra("Polars su Parquet", con_polars, "parquet")
risultato = cronometra("DuckDB su Parquet", con_duckdb, "parquet")

## 8 · Leggiamo i risultati

Raccogliamo i tempi in una tabella. L'ultima colonna dice quante volte ogni prova è stata più veloce della prima, cioè di pandas sul file CSV.

In [ ]:
tabella_tempi = pd.DataFrame({"prova": list(tempi.keys()), "secondi": list(tempi.values())})

# Rapporto fra il tempo di riferimento (pandas su CSV) e il tempo di ogni prova
tabella_tempi["volte più veloce di pandas su CSV"] = (tempi["pandas su CSV"] / tabella_tempi["secondi"]).round(1)
tabella_tempi["secondi"] = tabella_tempi["secondi"].round(2)

tabella_tempi

### Che cosa ci dicono questi numeri

- **Sul file CSV** il tempo è dominato dalla lettura del testo. Polars e DuckDB sono più rapidi di pandas perché distribuiscono la lettura su tutti i core e leggono solo ciò che serve alla domanda.
- **Sul file Parquet** tutti e tre gli strumenti migliorano nettamente, pandas compreso. Spesso il guadagno più grande non viene dal cambio di strumento ma dal cambio di **formato**.
- **La risposta è la stessa** in tutti i casi: cambiano i tempi, non i risultati.

### Indicazioni pratiche

| Situazione | Strumento indicato | Motivo |
|---|---|---|
| Fino a qualche milione di righe, con molte regole di controllo e pulizia | **pandas** | È lo strumento più diffuso e più documentato; a questi volumi i tempi sono già di pochi secondi. |
| Decine o centinaia di milioni di righe, su una sola macchina | **Polars** oppure **DuckDB** | Usano tutti i core, leggono solo ciò che serve e sanno elaborare i dati a blocchi, senza caricarli tutti in memoria. |
| Gruppo di lavoro che ragiona già in SQL | **DuckDB** | La richiesta si scrive in SQL direttamente sui file: le competenze esistenti si riutilizzano subito. |
| Dati che si trovano già in un database aziendale | **il database stesso** | Conviene far eseguire filtri, unioni e aggregazioni al database, con una istruzione SQL, e usare Python per orchestrare e controllare. |
| File intermedi scambiati fra un passo e l'altro del flusso | **formato Parquet** | Più compatto del CSV, molto più rapido da leggere e con i tipi delle colonne già definiti. |

### Tre avvertenze

1. **Non è una scelta esclusiva.** I tre strumenti convivono nello stesso flusso: il risultato di DuckDB diventa un DataFrame di pandas con `.df()`, quello di Polars con `.to_pandas()`. Si può usare il motore più veloce solo nel passo più pesante e lasciare tutto il resto in pandas.
2. **Prima misurare, poi cambiare.** Ha senso introdurre un nuovo strumento solo quando un passo del flusso è davvero lento. Per saperlo bisogna misurarlo, come abbiamo fatto qui.
3. **Oltre la singola macchina.** Quando i dati superano la capacità di un solo computer servono sistemi distribuiti, che ripartiscono il lavoro su più macchine. È un tema diverso, che resta fuori da questa demo.

### Per approfondire

- Novità di pandas 3.0: <https://pandas.pydata.org/docs/whatsnew/v3.0.0.html>
- Guida al passaggio a Polars 2.0: <https://docs.pola.rs/releases/upgrade/2/>
- DuckDB in Python: <https://duckdb.org/docs/2.0/clients/python/overview.html>